In [1]:
import sqlite3
import pandas as pd
import os

# =============================================
# CONNECT TO YOUR DATABASE
# =============================================
DB_PATH = "E:/Sidra/JupyterProjects/RiyalPay-Project/riyalpay.db"

# Check if database exists
if not os.path.exists(DB_PATH):
    print(f"❌ Database not found at: {DB_PATH}")
    print("Please make sure the database exists!")
else:
    print(f"✅ Database found at: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
print("✅ Connected to database successfully!\n")

# =============================================
# EXPERIMENT ANALYSIS
# =============================================

print("=" * 80)
print("🔬 EXPERIMENT ANALYSIS: exp_kyc_streamline_2026")
print("=" * 80)
print()

# ---------------------------------------------------------------------
# Query 1: SAMPLE RATIO MISMATCH (SRM) CHECK
# ---------------------------------------------------------------------
print("📋 1. SAMPLE RATIO MISMATCH (SRM) CHECK")
print("-" * 80)
print("Expect: iOS close to 50/50, Android meaningfully skewed")
print()

query1 = """
SELECT platform, variant, COUNT(*) AS n_users
FROM clean_experiment_assignments
WHERE is_contaminated = 0
GROUP BY platform, variant
ORDER BY platform, variant;
"""

df1 = pd.read_sql(query1, conn)
print(df1.to_string(index=False))
print()

# Calculate and display percentages
if len(df1) > 0:
    print("📊 Split percentages by platform:")
    for platform in df1['platform'].unique():
        platform_data = df1[df1['platform'] == platform]
        if len(platform_data) == 2:
            total = platform_data['n_users'].sum()
            for _, row in platform_data.iterrows():
                pct = round(row['n_users'] * 100 / total, 1)
                print(f"  {platform} - {row['variant']}: {pct}% ({row['n_users']:,} users)")
        else:
            print(f"  {platform}: Only {len(platform_data)} variant(s) found!")
print()

# ---------------------------------------------------------------------
# Query 2: CONTAMINATION SUMMARY
# ---------------------------------------------------------------------
print("📋 2. CONTAMINATION SUMMARY")
print("-" * 80)

query2 = """
SELECT
  COUNT(*) AS total_assigned_users,
  SUM(is_contaminated) AS contaminated_users,
  ROUND(SUM(is_contaminated) * 100.0 / COUNT(*), 2) AS pct_contaminated
FROM clean_experiment_assignments;
"""

df2 = pd.read_sql(query2, conn)
print(df2.to_string(index=False))
print()

# Interpret contamination
if len(df2) > 0:
    total = df2.iloc[0]['total_assigned_users']
    contaminated = df2.iloc[0]['contaminated_users']
    pct = df2.iloc[0]['pct_contaminated']
    
    print(f"📊 Interpretation:")
    print(f"   Total users: {total:,}")
    print(f"   Contaminated users: {contaminated:,} ({pct}%)")
    
    if pct > 5:
        print("   ⚠️  WARNING: More than 5% contaminated users!")
        print("   → Consider excluding contaminated users from analysis")
    elif pct > 1:
        print("   ⚠️  Note: {pct}% contaminated users")
        print("   → Excluding contaminated users is recommended")
    else:
        print("   ✅ Good: Low contamination rate")
print()

# ---------------------------------------------------------------------
# Query 3: PRIMARY OUTCOME - KYC Approval Rate by Arm
# ---------------------------------------------------------------------
print("📋 3. PRIMARY OUTCOME: KYC Approval Rate by Arm")
print("-" * 80)

query3 = """
WITH kyc_outcome AS (
  SELECT user_id, MAX(CASE WHEN event_name = 'kyc_approved' THEN 1 ELSE 0 END) AS approved
  FROM clean_events
  WHERE is_suspected_bot = 0
  GROUP BY user_id
)
SELECT
  ea.platform,
  ea.variant,
  COUNT(*) AS n_assigned,
  SUM(COALESCE(k.approved, 0)) AS n_approved,
  ROUND(SUM(COALESCE(k.approved, 0)) * 100.0 / COUNT(*), 2) AS approval_rate_pct
FROM clean_experiment_assignments ea
LEFT JOIN kyc_outcome k ON k.user_id = ea.user_id
WHERE ea.is_contaminated = 0
GROUP BY ea.platform, ea.variant
ORDER BY ea.platform, ea.variant;
"""

df3 = pd.read_sql(query3, conn)
print(df3.to_string(index=False))
print()

# Calculate lift for each platform
if len(df3) > 0:
    print("📊 Lift (Improvement) by Platform:")
    for platform in df3['platform'].unique():
        platform_data = df3[df3['platform'] == platform]
        if len(platform_data) == 2:
            # Get control and test rates
            control_rate = platform_data[platform_data['variant'] == 'control']['approval_rate_pct'].values[0]
            treatment_rate = platform_data[platform_data['variant'] == 'treatment']['approval_rate_pct'].values[0]
            lift = treatment_rate - control_rate
            lift_pct = round(lift / control_rate * 100, 1) if control_rate > 0 else 0
            
            print(f"  {platform}:")
            print(f"    Control: {control_rate}%")
            print(f"    Treatment:    {treatment_rate}%")
            print(f"    Lift:    +{lift:.1f} percentage points (+{lift_pct}%)")
            
            if lift > 0:
                print(f"    ✅ Positive improvement")
            else:
                print(f"    ⚠️  No improvement or negative")
    print()

# ---------------------------------------------------------------------
# Query 4: SECONDARY OUTCOME - Activation Rate by Arm
# ---------------------------------------------------------------------
print("📋 4. SECONDARY OUTCOME: Activation Rate by Arm")
print("-" * 80)

query4 = """
WITH activated AS (
  SELECT DISTINCT user_id FROM clean_transactions WHERE status = 'success'
)
SELECT
  ea.platform,
  ea.variant,
  COUNT(*) AS n_assigned,
  COUNT(a.user_id) AS n_activated,
  ROUND(COUNT(a.user_id) * 100.0 / COUNT(*), 2) AS activation_rate_pct
FROM clean_experiment_assignments ea
LEFT JOIN activated a ON a.user_id = ea.user_id
WHERE ea.is_contaminated = 0
GROUP BY ea.platform, ea.variant
ORDER BY ea.platform, ea.variant;
"""

df4 = pd.read_sql(query4, conn)
print(df4.to_string(index=False))
print()

# Calculate lift for activation
if len(df4) > 0:
    print("📊 Activation Lift by Platform:")
    for platform in df4['platform'].unique():
        platform_data = df4[df4['platform'] == platform]
        if len(platform_data) == 2:
            control_rate = platform_data[platform_data['variant'] == 'control']['activation_rate_pct'].values[0]
            treatment_rate = platform_data[platform_data['variant'] == 'treatment']['activation_rate_pct'].values[0]
            lift = treatment_rate - control_rate
            lift_pct = round(lift / control_rate * 100, 1) if control_rate > 0 else 0
            
            print(f"  {platform}:")
            print(f"    Control: {control_rate}%")
            print(f"    Treatment:    {treatment_rate}%")
            print(f"    Lift:    +{lift:.1f} percentage points (+{lift_pct}%)")
            
            if lift > 0:
                print(f"    ✅ Positive improvement")
            else:
                print(f"    ⚠️  No improvement or negative")
    print()

# ---------------------------------------------------------------------
# Query 5: WEEKLY KYC APPROVAL RATE (Novelty Effect Check)
# ---------------------------------------------------------------------
print("📋 5. WEEKLY KYC APPROVAL RATE (Novelty Effect Check)")
print("-" * 80)
print("Checks if the treatment's lift shrinks over the 6-week window")
print()

query5 = """
WITH kyc_outcome AS (
  SELECT user_id, MIN(CASE WHEN event_name = 'kyc_approved' THEN event_timestamp END) AS approved_ts
  FROM clean_events
  WHERE is_suspected_bot = 0
  GROUP BY user_id
)
SELECT
  ea.variant,
  CAST(((JULIANDAY(ea.assignment_date) - JULIANDAY('2026-02-02')) / 7) AS INT) + 1 AS experiment_week,
  COUNT(*) AS n_assigned,
  SUM(CASE WHEN k.approved_ts IS NOT NULL THEN 1 ELSE 0 END) AS n_approved,
  ROUND(SUM(CASE WHEN k.approved_ts IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS approval_rate_pct
FROM clean_experiment_assignments ea
LEFT JOIN kyc_outcome k ON k.user_id = ea.user_id
WHERE ea.is_contaminated = 0
GROUP BY ea.variant, experiment_week
ORDER BY experiment_week, ea.variant;
"""

df5 = pd.read_sql(query5, conn)
print(df5.to_string(index=False))
print()

# Create a pivot table for weekly comparison
if len(df5) > 0:
    # Pivot to compare control vs test by week
    pivot_weekly = df5.pivot(index='experiment_week', columns='variant', values='approval_rate_pct')
    
    if not pivot_weekly.empty:
        print("📊 Weekly Approval Rate Comparison:")
        if 'control' in pivot_weekly.columns and 'treatment' in pivot_weekly.columns:
            pivot_weekly['lift'] = pivot_weekly['treatment'] - pivot_weekly['control']
            print(pivot_weekly.to_string())
            
            # Check for novelty effect
            print("\n📊 Novelty Effect Check:")
            first_lift = pivot_weekly['lift'].iloc[0] if len(pivot_weekly) > 0 else None
            last_lift = pivot_weekly['lift'].iloc[-1] if len(pivot_weekly) > 0 else None
            
            if first_lift is not None and last_lift is not None:
                print(f"   Week 1 lift: {first_lift:.2f} percentage points")
                print(f"   Week {len(pivot_weekly)} lift: {last_lift:.2f} percentage points")
                
                if last_lift < first_lift * 0.5:
                    print("   ⚠️  NOVELTY EFFECT DETECTED: Lift shrinking over time")
                    print("   → The initial improvement may be due to novelty")
                else:
                    print("   ✅ No strong novelty effect detected")
                    print("   → The improvement appears consistent")
    print()

# =============================================
# SUMMARY
# =============================================
print("=" * 80)
print("📊 EXPERIMENT ANALYSIS SUMMARY")
print("=" * 80)
print()

# Check if experiment results are reliable
print("🔍 Key Findings:")

# 1. Check contamination
if len(df2) > 0:
    pct_contaminated = df2.iloc[0]['pct_contaminated']
    if pct_contaminated < 1:
        print("✅ Low contamination rate - experiment assignment is clean")
    else:
        print(f"⚠️  {pct_contaminated}% contamination rate - consider excluding contaminated users")

# 2. Check SRM
if len(df1) > 0:
    srm_issues = False
    for platform in df1['platform'].unique():
        platform_data = df1[df1['platform'] == platform]
        if len(platform_data) == 2:
            total = platform_data['n_users'].sum()
            for _, row in platform_data.iterrows():
                pct = round(row['n_users'] * 100 / total, 1)
                if abs(pct - 50) > 5:
                    print(f"⚠️  Sample Ratio Mismatch on {platform}: {pct}% in {row['variant']}")
                    srm_issues = True
    if not srm_issues:
        print("✅ Good 50/50 split across all platforms")

# 3. Check KYC lift
if len(df3) > 0:
    for platform in df3['platform'].unique():
        platform_data = df3[df3['platform'] == platform]
        if len(platform_data) == 2:
            control_rate = platform_data[platform_data['variant'] == 'control']['approval_rate_pct'].values[0]
            treatment_rate = platform_data[platform_data['variant'] == 'treatment']['approval_rate_pct'].values[0]
            lift = treatment_rate - control_rate
            if lift > 0:
                print(f"✅ {platform}: KYC approval improved by {lift:.1f} percentage points")
            else:
                print(f"⚠️  {platform}: No improvement in KYC approval")

print()
print("=" * 80)
print("✅ EXPERIMENT ANALYSIS COMPLETE!")
print("=" * 80)

# =============================================
# CLOSE CONNECTION
# =============================================
conn.close()

✅ Database found at: riyalpay.db
✅ Connected to database successfully!

🔬 EXPERIMENT ANALYSIS: exp_kyc_streamline_2026

📋 1. SAMPLE RATIO MISMATCH (SRM) CHECK
--------------------------------------------------------------------------------
Expect: iOS close to 50/50, Android meaningfully skewed

platform   variant  n_users
 Android   control     2319
 Android treatment     1420
     iOS   control     1055
     iOS treatment     1175

📊 Split percentages by platform:
  Android - control: 62.0% (2,319 users)
  Android - treatment: 38.0% (1,420 users)
  iOS - control: 47.3% (1,055 users)
  iOS - treatment: 52.7% (1,175 users)

📋 2. CONTAMINATION SUMMARY
--------------------------------------------------------------------------------
 total_assigned_users  contaminated_users  pct_contaminated
                 6012                  43              0.72

📊 Interpretation:
   Total users: 6,012.0
   Contaminated users: 43.0 (0.72%)
   ✅ Good: Low contamination rate

📋 3. PRIMARY OUTCOME: KYC 